### Library & Dataset import

In [ ]:
pip install statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error

# Set plot style and size
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
!git clone "https://github.com/GeeksforgeeksDS/21-Days-21-Projects-Dataset"

In [ ]:
df = pd.read_csv('/content/21-Days-21-Projects-Dataset/Datasets/airline_passenger_timeseries.csv')
df.head(5)

### EDA

In [ ]:
df.plot()
plt.title('Monthly Airline Passengers (1949-1960)')
plt.xlabel('Year')
plt.ylabel('Number of Passengers')
plt.show()

In [ ]:
# Convert 'Month' to datetime and set as index
df['Month'] = pd.to_datetime(df['Month'])
df.set_index('Month', inplace=True)

# Decompose the time series to visualize its components
decomposition = sm.tsa.seasonal_decompose(df['Passengers'], model='multiplicative')

fig = decomposition.plot()
fig.set_size_inches(14, 10)
plt.show()

In [ ]:
def test_stationarity(timeseries):
    # Perform Dickey-Fuller test:
    print('Results of Dickey-Fuller Test:')
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print(dfoutput)

test_stationarity(df['Passengers'])

In [ ]:
# 1. Apply log transformation to stabilize the variance
df_log = np.log(df['Passengers'])

# 2. Apply differencing to remove the trend
df_diff = df_log.diff().dropna()

# Plot the stationary series
df_diff.plot()
plt.title('Stationary Time Series (Log-Differenced)')
plt.show()

# Retest for stationarity
test_stationarity(df_diff)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
plot_acf(df_diff, ax=ax1, lags=20)
plot_pacf(df_diff, ax=ax2, lags=20)
plt.show()

In [ ]:
# Split data into training and test sets
train_data = df_log[:'1958']
test_data = df_log['1959':]

# Build ARIMA model
model = ARIMA(train_data, order=(1, 1, 1), freq='MS')
arima_result = model.fit()

# Get forecast
forecast = arima_result.get_forecast(steps=len(test_data))
forecast_ci = forecast.conf_int()

# Plot the forecast
plt.figure(figsize=(14, 7))
plt.plot(df_log, label='Original Log Data')
plt.plot(forecast.predicted_mean, label='Forecast')
plt.fill_between(forecast_ci.index, forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1], color='k', alpha=.15)
plt.title('ARIMA Model Forecast')
plt.legend()
plt.show()

In [ ]:
# Build SARIMA model
# We can find the optimal P, D, Q through a grid search, but common values are 1.
sarima_model = sm.tsa.statespace.SARIMAX(train_data,
                                          order=(1, 1, 1),
                                          seasonal_order=(1, 1, 1, 12),
                                          enforce_stationarity=False,
                                          enforce_invertibility=False,
                                          freq='MS') # Explicitly set frequency to suppress warnings
sarima_result = sarima_model.fit()

# Get forecast
sarima_forecast = sarima_result.get_forecast(steps=len(test_data))
sarima_forecast_ci = sarima_forecast.conf_int()

# Plot the forecast
plt.figure(figsize=(14, 7))
plt.plot(df_log, label='Original Log Data')
plt.plot(sarima_forecast.predicted_mean, label='SARIMA Forecast', color='red')
plt.fill_between(sarima_forecast_ci.index, sarima_forecast_ci.iloc[:, 0], sarima_forecast_ci.iloc[:, 1], color='r', alpha=.15)
plt.title('SARIMA Model Forecast')
plt.legend()
plt.show()

In [ ]:
# Reverse the log transformation to get actual passenger numbers
original_test_data = np.exp(test_data)
sarima_predictions = np.exp(sarima_forecast.predicted_mean)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(original_test_data, sarima_predictions))
print(f"SARIMA Model RMSE: {rmse:.2f}")

# Plot final results
plt.figure(figsize=(14, 7))
plt.plot(df['Passengers'], label='Original Data')
plt.plot(sarima_predictions, label='SARIMA Forecast', color='red')
plt.title('Final Forecast vs. Actual Data')
plt.legend()
plt.show()

## ASSIGNMENT SUBMISSION

### Q1

1. **Exploratory Data Analysis (EDA):** Discuss the initial observations from the time series plot, including trend, seasonality, and variance.

Initial Observations from the Time Series Plot:

Trend: The time series shows a clear upward trend, indicating that the number of airline passengers steadily increased over the period from 1949 to 1960.

Seasonality: There are repeating patterns in the data, specifically visible peaks and troughs. This suggests the presence of seasonality, possibly due to periodic fluctuations in airline traffic, such as holiday travel or other seasonal events.

Variance: The variance in the data appears to increase over time. Initially, the number of passengers is lower, but as time progresses, the variation in passenger numbers becomes more significant. This suggests that there might be a need to stabilize the variance for better modeling.

### Q2
2. **Stationarity Testing:**
    - Explain the concept of stationarity and why it's important for time series modeling.
    - Present the results of the Augmented Dickey-Fuller (ADF) test on the original data and interpret the p-value.
    - Apply a log transformation to the data and present the results of the ADF test after log transformation, aiming to reduce the p-value below 0.05. Discuss your findings.
    - Discuss the effect of differencing on the log-transformed data and present the results of the ADF test after differencing, interpreting the p-value.

Explanation of Stationarity:

Stationarity means that the statistical properties of the time series, such as mean, variance, and autocorrelation, do not change over time. A stationary series is important because many time series models, including ARIMA, assume stationarity.

A time series needs to be stationary for reliable predictions because non-stationary data might lead to biased estimates and overfitting.

Augmented Dickey-Fuller (ADF) Test:
The ADF test is used to check if a time series is stationary. A null hypothesis of the ADF test is that the series has a unit root (i.e., it is non-stationary). We want the p-value to be less than 0.05 to reject the null hypothesis and conclude that the series is stationary.

Original Data:
Let's take a look at the results of the ADF test for the original data:

If the p-value is greater than 0.05, the series is non-stationary. The initial plot of the time series indicated a clear upward trend and seasonal patterns, so we expect the data to be non-stationary.

Log Transformation:
Applying a log transformation helps stabilize the variance, particularly when there's increasing variance over time (which we observed in the EDA).

After applying the log transformation, the series should have more stable variance. We then perform the ADF test again on the log-transformed data.

Differencing:
After log transformation, we can perform differencing to remove trends and achieve stationarity. Differencing helps to remove trends by subtracting the previous value from the current value.

The goal is to make the series stationary by reducing autocorrelation and making the mean constant.

### Q3
3. **ARIMA Model Performance:** Based on the stationarity test results after log transformation (before differencing), discuss whether you would expect a non-seasonal ARIMA model to perform well on the log-transformed data. Build and evaluate a non-seasonal ARIMA model on the log-transformed data (without differencing) and compare its performance to the SARIMA model built later in the notebook.

Discussion of ARIMA on Log-Transformed Data:
Based on the log-transformed data (before differencing), we expect that the series might still exhibit non-stationarity due to seasonality. Therefore, a non-seasonal ARIMA model might not perform well because it doesn't explicitly account for seasonality. A SARIMA model, which handles seasonality, is more suitable.

ARIMA Model Evaluation:
After building an ARIMA model on the log-transformed data, we'll evaluate its performance and compare it with the SARIMA model built later in the notebook. The performance can be assessed using metrics like RMSE (Root Mean Squared Error).

## EDA/Feature Engineering

In [ ]:
# Plot the original data
df.plot()
plt.title('Monthly Airline Passengers (1949-1960)')
plt.xlabel('Year')
plt.ylabel('Number of Passengers')
plt.show()

# Step 1: Seasonal Decomposition (to see seasonality and trends)
decomposition = sm.tsa.seasonal_decompose(df['Passengers'], model='multiplicative')
fig = decomposition.plot()
fig.set_size_inches(14, 10)
plt.show()

# Step 2: Remove the seasonal component from the series to get detrended data
detrended_data = df['Passengers'] / decomposition.seasonal
detrended_data.plot()
plt.title('Detrended Data')
plt.show()

# Step 3: Differencing the detrended data to remove the trend
detrended_diff = detrended_data.diff().dropna()
detrended_diff.plot()
plt.title('Differenced Detrended Data')
plt.show()

# After all transformations (detrending, differencing, log, etc.)
# Ensure index is preserved during transformations

# Step 4: Add 1 and apply log transformation
detrended_diff_safe = detrended_diff + 1
log_transformed = np.log(detrended_diff_safe)

# Clean invalid values (Inf or NaN)
log_transformed.replace([np.inf, -np.inf], np.nan, inplace=True)
log_transformed.dropna(inplace=True)

# At this point, `log_transformed` should be a valid time series with a proper datetime index

# Check index and data before splitting
print(log_transformed.head())
print(log_transformed.index)

# STEP: Safe train/test split
train_data = log_transformed[:'1958']
test_data = log_transformed['1959':]

# Ensure both have same frequency and valid indexes
train_data = train_data.asfreq('MS')
test_data = test_data.asfreq('MS')

# Now fit ARIMA model
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(train_data, order=(1, 1, 1))  # Don't pass 'freq' here; use .asfreq() instead
arima_result = model.fit()

# Forecasting
forecast = arima_result.get_forecast(steps=len(test_data))
forecast_ci = forecast.conf_int()

# Plot the forecast
plt.figure(figsize=(14, 7))
plt.plot(log_transformed, label='Log Transformed Series')
plt.plot(forecast.predicted_mean, label='ARIMA Forecast')
plt.fill_between(forecast_ci.index, forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1], color='k', alpha=.15)
plt.title('ARIMA Model Forecast (Log Transformed Data)')
plt.legend()
plt.show()

# Step 5: Augmented Dickey-Fuller Test for stationarity
def test_stationarity(timeseries):
    print('Results of Dickey-Fuller Test:')
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print(dfoutput)

# Run ADF test after log transformation
test_stationarity(log_transformed)

# Step 6: Differencing the log-transformed data to make it stationary
log_diff = log_transformed.diff().dropna()
log_diff.plot()
plt.title('Differenced Log Transformed Data')
plt.show()

# Run ADF test after differencing the log-transformed data
test_stationarity(log_diff)

# Step 7: Plot ACF and PACF to determine ARIMA parameters
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
plot_acf(log_diff, ax=ax1, lags=20)
plot_pacf(log_diff, ax=ax2, lags=20)
plt.show()

# Step 8: ARIMA model fitting (without seasonal components yet)
from statsmodels.tsa.arima.model import ARIMA
train_data = log_transformed[:'1958']
test_data = log_transformed['1959':]

# ARIMA model
model = ARIMA(train_data, order=(1, 1, 1))
arima_result = model.fit()

# Forecasting
forecast = arima_result.get_forecast(steps=len(test_data))
forecast_ci = forecast.conf_int()

# Plot the forecast
plt.figure(figsize=(14, 7))
plt.plot(log_transformed, label='Original Log Data')
plt.plot(forecast.predicted_mean, label='ARIMA Forecast')
plt.fill_between(forecast_ci.index, forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1], color='k', alpha=.15)
plt.title('ARIMA Model Forecast (Log Transformed Data)')
plt.legend()
plt.show()


After applying seasonal decomposition, removing the seasonal component, differencing, and log transformation, the Augmented Dickey-Fuller test returned a p-value of 1.35e-15. Since this is significantly below the 0.05 threshold, we conclude that the transformed time series is stationary and suitable for ARIMA modeling.